In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip -q install bertopic sentence-transformers pymupdf jieba nltk openpyxl scikit-learn pandas matplotlib

In [ ]:
import os
import re
import glob
import fitz
import jieba
import nltk
import pandas as pd
import numpy as np

from collections import Counter

print("Libraries loaded.")

/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.13/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


Libraries loaded.


In [ ]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
PDF_FOLDER = "/content/drive/MyDrive/chromium"

pdf_files = glob.glob(os.path.join(PDF_FOLDER, "*.pdf"))

print(f"Found {len(pdf_files)} PDF files.")

for f in pdf_files:
    print(os.path.basename(f))

Found 67 PDF files.
Current supply status of chromium resources in China.pdf
大国博弈背景下中亚国家的关键矿产战略_张含睿.pdf
哈萨克斯坦主要矿产资源特征及矿业投资环境_高永伟.pdf
新一轮大国竞争背景下美国的中亚战略_戴惟静.pdf
能源转型背景下关键矿产博弈与中国供应安全_王永中.pdf
CRITICAL MINERALS SUPPLY CHAINS AND U.S. STRATEGIC INVESTMENT IN CENTRAL ASIA’S MINING SECTOR.pdf
我国铬铁行业发展概况及现状_李建法-1.pdf
世界政治极端化与国际关键矿产供应链安全_陈小鼎.pdf
共建“一带一路”框架内中国—中亚供应链安全合作_李孝天.pdf
全球供应链重构视阈下美欧的中亚关键矿产政策_张建.pdf
哈萨克斯坦的关键矿产开发及国际合作战略动向_杨进.pdf
我国关键矿产原料进口供应链安全风险及应对_韩亚品.pdf
美国重构关键矿产供应链与“一带一路”关键矿产合作高质量发展_孙海泳.pdf
An Overview of Applied Ferrochrome Production.pdf
Central Asia is a missing link in analyses of critical materials for the global clean energy transition.pdf
Critical Mineral Strategies of Kazakhstan and Uzbekistan.pdf
CRITICAL MINERAL SUPPLY CHAIN SECURITY AND MULTI-LAYERED REGIONAL GOVERNANCE INSIGHTS FROM CENTRAL ASIA.pdf
Competition and Cooperation in the Critical Minerals Sector under the Central Asia C5+1 Mechanisms.pdf
Central Asia in the EU’s Critical Raw Materials Strategy.pdf
Kazakhstan’s t

In [ ]:
def detect_language_robust(text):

    # Take the first ~10,000 characters
    sample = text[:10000]

    # Chinese characters
    chinese_chars = re.findall(
        r'[\u4e00-\u9fff]',
        sample
    )

    # English alphabetic characters
    english_chars = re.findall(
        r'[A-Za-z]',
        sample
    )

    chinese_count = len(chinese_chars)
    english_count = len(english_chars)

    total = chinese_count + english_count

    if total == 0:
        return "Unknown"

    chinese_ratio = chinese_count / total

    if chinese_ratio >= 0.30:
        return "Chinese"

    else:
        return "English"

In [ ]:
papers_df["language"] = papers_df["text"].apply(
    detect_language_robust
)

In [ ]:
print(papers_df["language"].value_counts())

display(
    papers_df[
        ["paper_id", "filename", "language"]
    ]
)

language
English    34
Chinese    33
Name: count, dtype: int64


,paper_id,filename,language
0,P001,Current supply status of chromium resources in...,English
1,P002,大国博弈背景下中亚国家的关键矿产战略_张含睿.pdf,Chinese
2,P003,哈萨克斯坦主要矿产资源特征及矿业投资环境_高永伟.pdf,Chinese
3,P004,新一轮大国竞争背景下美国的中亚战略_戴惟静.pdf,Chinese
4,P005,能源转型背景下关键矿产博弈与中国供应安全_王永中.pdf,Chinese
...,...,...,...
62,P063,Securing Critical Minerals Supply Chains U.S. ...,English
63,P064,"Critical Minerals, Mining Sector Investment, a...",English
64,P065,Assessing the supply risk of geopolitics on cr...,English
65,P066,Resource curse in the age of critical minerals...,English


In [ ]:
def extract_pdf_text(pdf_path):

    doc = fitz.open(pdf_path)

    pages = []

    for page in doc:
        text = page.get_text("text")
        pages.append(text)

    doc.close()

    return "\n".join(pages)

In [ ]:
papers = []

for i, pdf_path in enumerate(pdf_files, start=1):

    filename = os.path.basename(pdf_path)

    # First extract the actual paper text
    text = extract_pdf_text(pdf_path)

    # Then identify language from the text
    language = detect_language_robust(text)

    papers.append({
        "paper_id": f"P{i:03d}",
        "filename": filename,
        "language": language,
        "text": text
    })

papers_df = pd.DataFrame(papers)

In [ ]:
papers_df[["paper_id", "filename", "language"]]

,paper_id,filename,language
0,P001,Current supply status of chromium resources in...,English
1,P002,大国博弈背景下中亚国家的关键矿产战略_张含睿.pdf,Chinese
2,P003,哈萨克斯坦主要矿产资源特征及矿业投资环境_高永伟.pdf,Chinese
3,P004,新一轮大国竞争背景下美国的中亚战略_戴惟静.pdf,Chinese
4,P005,能源转型背景下关键矿产博弈与中国供应安全_王永中.pdf,Chinese
...,...,...,...
62,P063,Securing Critical Minerals Supply Chains U.S. ...,English
63,P064,"Critical Minerals, Mining Sector Investment, a...",English
64,P065,Assessing the supply risk of geopolitics on cr...,English
65,P066,Resource curse in the age of critical minerals...,English


### Separate documents by language

Based on the language detection, we'll split the documents into English and Chinese subsets for topic modeling.

In [ ]:
english_papers_df = papers_df[papers_df['language'] == 'English'].copy()
chinese_papers_df = papers_df[papers_df['language'] == 'Chinese'].copy()

print(f"Number of English documents: {len(english_papers_df)}")
print(f"Number of Chinese documents: {len(chinese_papers_df)}")

display(english_papers_df.head(2))
display(chinese_papers_df.head(2))

Number of English documents: 34
Number of Chinese documents: 33


,paper_id,filename,language,text
0,P001,Current supply status of chromium resources in...,English,Green and Smart Mining Engineering 1 (2024) 53...
5,P006,CRITICAL MINERALS SUPPLY CHAINS AND U.S. STRAT...,English,International Multidisciplinary Research in Ac...


,paper_id,filename,language,text
1,P002,大国博弈背景下中亚国家的关键矿产战略_张含睿.pdf,Chinese,大国博弈背景下中亚国家的关键矿产战略 \n大国博弈背景下中亚国家的关键矿产战略∗\n张含睿\...
2,P003,哈萨克斯坦主要矿产资源特征及矿业投资环境_高永伟.pdf,Chinese,第５８卷 第２期\n２０２２年３月\n地质与勘探\nＧＥＯＬＯＧＹＡＮＤＥＸＰＬＯＲＡＴＩＯ...


### BERTopic for English Documents

Now, let's apply BERTopic to the English documents. We will use a multilingual Sentence Transformer model that performs well across different languages.

In [ ]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# Initialize a multilingual sentence transformer model
sentence_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Initialize BERTopic
# We will use the 'english' language option for c-TF-IDF (term weighting)
# For Chinese, we'll use 'chinese-simplified'
topic_model_english = BERTopic(language="english", calculate_probabilities=True, verbose=True,
                               embedding_model=sentence_model)

# Fit BERTopic to the English documents
english_topics, english_probs = topic_model_english.fit_transform(english_papers_df['text'].tolist())

# Display the top 10 topics for English documents
topic_model_english.get_topic_info().head(10)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-09-05 12:15:22,707 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

2026-09-05 12:15:29,761 - BERTopic - Embedding - Completed ✓
2026-09-05 12:15:29,764 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-05 12:15:43,475 - BERTopic - Dimensionality - Completed ✓
2026-09-05 12:15:43,478 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-05 12:15:43,491 - BERTopic - Cluster - Completed ✓
2026-09-05 12:15:43,502 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-05 12:15:44,409 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,34,-1_the_of_and_in,"[the, of, and, in, to, for, is, as, china, on]",[Kazakhstan: An Aspiring Middle \nPower in the...


### Stopword Removal for English Documents

To further refine the English topics, we will remove common English stop words from the documents. This can help BERTopic focus on more significant terms when identifying topics.

In [ ]:
nltk.download('stopwords')
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words('english'))

def remove_stopwords(text):
    words = text.lower().split()
    filtered_words = [word for word in words if word not in english_stopwords]
    return ' '.join(filtered_words)

english_papers_df['text_no_stopwords'] = english_papers_df['text'].apply(remove_stopwords)

print("English documents cleaned of stopwords.")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


English documents cleaned of stopwords.


### Aggressive Text Cleaning for English Documents

To further reduce noise in collocations, we will apply a more aggressive cleaning step to the English documents. This includes removing punctuation, numbers, and tokens shorter than a certain length (e.g., 2 characters) from the stopword-removed text.

In [ ]:
def aggressive_clean_english(text):
    # Remove punctuation and special characters, keep only letters and spaces
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Convert to lowercase and split into words
    words = text.lower().split()
    # Filter out very short words (e.g., single letters or two-letter words that might be noise)
    filtered_words = [word for word in words if len(word) > 2]
    return ' '.join(filtered_words)

english_papers_df['text_cleaned'] = english_papers_df['text_no_stopwords'].apply(aggressive_clean_english)

print("English documents aggressively cleaned.")


English documents aggressively cleaned.


### Re-running BERTopic with Stopword-Removed English Documents

Now, let's re-apply BERTopic to the English documents, this time using the texts where stop words have been removed. We will keep `nr_topics=7` and further reduce `min_topic_size` to 3, as previously discussed, to potentially capture even smaller, more distinct topics.

In [ ]:
# Re-initialize BERTopic for English with refined parameters and stopword-removed text
# Using min_topic_size=3 to allow for very small, potentially niche topics
topic_model_english_refined_stopwords = BERTopic(language="english", calculate_probabilities=True, verbose=True,
                                                 embedding_model=sentence_model,
                                                 nr_topics=7, # Keep target number of topics
                                                 min_topic_size=3 # Further reduce min topic size
                                                )

# Fit the refined BERTopic model to the English documents with stopwords removed
english_topics_refined_stopwords, english_probs_refined_stopwords = topic_model_english_refined_stopwords.fit_transform(english_papers_df['text_no_stopwords'].tolist())

print("Refined BERTopic for English Documents (stopwords removed, min_topic_size=3):")
display(topic_model_english_refined_stopwords.get_topic_info())

2026-09-05 12:21:43,138 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

2026-09-05 12:21:52,169 - BERTopic - Embedding - Completed ✓
2026-09-05 12:21:52,171 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-05 12:21:52,302 - BERTopic - Dimensionality - Completed ✓
2026-09-05 12:21:52,303 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-05 12:21:52,319 - BERTopic - Cluster - Completed ✓
2026-09-05 12:21:52,321 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-09-05 12:21:53,163 - BERTopic - Representation - Completed ✓
2026-09-05 12:21:53,165 - BERTopic - Topic reduction - Reducing number of topics
2026-09-05 12:21:53,169 - BERTopic - Topic reduction - Number of topics (7) is equal or higher than the clustered topics(3).
2026-09-05 12:21:53,171 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-05 12:21:54,361 - BERTopic - Representation - Completed ✓


Refined BERTopic for English Documents (stopwords removed, min_topic_size=3):


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2,-1_central_asia_participant_kazakhstan,"[central, asia, participant, kazakhstan, forei...",[review central asia missing link analyses cri...
1,0,10,0_kazakhstan_nfne_abiotic_secondary,"[kazakhstan, nfne, abiotic, secondary, primary...",[online issn 2412-0731 406 r-economy.com r-eco...
2,1,22,1_china_minerals_critical_mineral,"[china, minerals, critical, mineral, supply, r...",[unsw law & justice research series demystifyi...


In [ ]:
print("Topics for English Documents (stopwords removed, min_topic_size=3):")
display(topic_model_english_refined_stopwords.get_topic_info())

Topics for English Documents (stopwords removed, min_topic_size=3):


,Topic,Count,Name,Representation,Representative_Docs
0,-1,2,-1_central_asia_participant_kazakhstan,"[central, asia, participant, kazakhstan, forei...",[review central asia missing link analyses cri...
1,0,10,0_kazakhstan_nfne_abiotic_secondary,"[kazakhstan, nfne, abiotic, secondary, primary...",[online issn 2412-0731 406 r-economy.com r-eco...
2,1,22,1_china_minerals_critical_mineral,"[china, minerals, critical, mineral, supply, r...",[unsw law & justice research series demystifyi...


### BERTopic for Chinese Documents

Next, we'll apply BERTopic to the Chinese documents. For Chinese text, it's beneficial to perform tokenization with a library like `jieba` before feeding it to BERTopic, especially when using models not specifically trained on raw Chinese characters without spaces. Although our Sentence Transformer is multilingual, `jieba` can help with term extraction for c-TF-IDF. We will also specify the `language="chinese-simplified"` for BERTopic to optimize c-TF-IDF for Chinese.

In [ ]:
# Tokenize Chinese text using jieba
def tokenize_chinese(text):
    return " ".join(jieba.cut(text))

chinese_papers_df['tokenized_text'] = chinese_papers_df['text'].apply(tokenize_chinese)

# Initialize BERTopic for Chinese
topic_model_chinese = BERTopic(language="chinese-simplified", calculate_probabilities=True, verbose=True,
                               embedding_model=sentence_model)

# Fit BERTopic to the Chinese documents (using tokenized text for better keyword extraction)
chinese_topics, chinese_probs = topic_model_chinese.fit_transform(chinese_papers_df['tokenized_text'].tolist())

# Display the top 10 topics for Chinese documents
topic_model_chinese.get_topic_info().head(10)

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 0.912 seconds.
DEBUG:jieba:Loading model cost 0.912 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.
2026-09-05 12:15:50,862 - BERTopic - Embedding - Transforming documents to embeddings.


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

2026-09-05 12:15:58,076 - BERTopic - Embedding - Completed ✓
2026-09-05 12:15:58,079 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-09-05 12:15:58,191 - BERTopic - Dimensionality - Completed ✓
2026-09-05 12:15:58,194 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-09-05 12:15:58,205 - BERTopic - Cluster - Completed ✓
2026-09-05 12:15:58,213 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-09-05 12:15:58,955 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,5,-1_哈萨克斯坦_国家_发展_需求,"[哈萨克斯坦, 国家, 发展, 需求, 主要, 国际, 社会, 对外, 战略, 总统]",[ \n 第 ５ ５ 卷 第 ３ 期 \n ２ ０ ２ ２ 年 （ 总 ２ ２ ５ 期 ）...
1,0,14,0_矿产_关键_美国_供应链,"[矿产, 关键, 美国, 供应链, 全球, 中国, 中亚, 合作, 安全, 战略]",[《 国际 论坛 》 2024 年 第 5 期 \n · 42 · \n 美国 重构...
2,1,14,1_经济_风险_中国_供应商,"[经济, 风险, 中国, 供应商, 资源, 发展, 研究, 中亚国家, 影响, 采购]",[Ｉ \n 碩士 学位 论文 \n  \n  \n Ｉ \n 修 \n  \n  \...


### Aggressive Text Cleaning for Chinese Documents

Similar to English, we will apply aggressive cleaning to the Chinese tokenized text. This will help remove remaining punctuation, numbers, and very short tokens that might still be present after initial tokenization, reducing noise in collocation analysis.

In [ ]:
def aggressive_clean_chinese(text):
    # Remove punctuation and special characters (keeping only Chinese characters, English letters, and spaces)
    # Chinese character range: \u4e00-\u9fff
    text = re.sub(r'[^一-鿿_a-zA-Z\s]', '', text)
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Split into words (jieba output is space-separated)
    words = text.split()
    # Filter out very short tokens (e.g., single characters or two-character words that might be noise)
    # We'll be careful here to not remove too many meaningful Chinese characters. Len > 1 is a good start.
    filtered_words = [word for word in words if len(word) > 1]
    return ' '.join(filtered_words)

chinese_papers_df['text_cleaned'] = chinese_papers_df['tokenized_text'].apply(aggressive_clean_chinese)

print("Chinese documents aggressively cleaned.")


Chinese documents aggressively cleaned.


### TF-IDF for English Documents

Now we will apply TF-IDF (Term Frequency-Inverse Document Frequency) to the English documents to find words that are important to a document in a collection or corpus. We will use the `text_no_stopwords` column.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer for English
# We will use the stopword-removed text
tfidf_vectorizer_english = TfidfVectorizer(max_features=1000) # Limit to top 1000 features for manageable output

# Fit and transform the English documents (without stopwords)
tfidf_matrix_english = tfidf_vectorizer_english.fit_transform(english_papers_df['text_no_stopwords'].tolist())

# Get feature names (words)
feature_names_english = tfidf_vectorizer_english.get_feature_names_out()

# Convert TF-IDF matrix to a DataFrame for easier viewing
tfidf_df_english = pd.DataFrame(tfidf_matrix_english.toarray(), columns=feature_names_english)

print("TF-IDF matrix for English documents created. Displaying top terms:")

# To display top TF-IDF terms per document or overall
# For simplicity, let's display the top 10 most frequent (highest mean TF-IDF) terms across all English documents
overall_tfidf_scores_english = tfidf_df_english.mean().sort_values(ascending=False)
print("\nTop 10 overall TF-IDF terms in English documents:")
display(overall_tfidf_scores_english.head(10))

TF-IDF matrix for English documents created. Displaying top terms:

Top 10 overall TF-IDF terms in English documents:


,0
china,0.150817
critical,0.143155
kazakhstan,0.142875
minerals,0.138667
central,0.101087
mineral,0.097477
supply,0.087629
mining,0.086400
asia,0.086223
2025,0.078482


### TF-IDF for Chinese Documents

Similarly, we will apply TF-IDF to the Chinese documents using their `tokenized_text` to identify distinctive vocabulary.

In [ ]:
# Initialize TF-IDF Vectorizer for Chinese
tfidf_vectorizer_chinese = TfidfVectorizer(max_features=1000) # Limit to top 1000 features

# Fit and transform the Chinese documents (tokenized)
tfidf_matrix_chinese = tfidf_vectorizer_chinese.fit_transform(chinese_papers_df['tokenized_text'].tolist())

# Get feature names (words)
feature_names_chinese = tfidf_vectorizer_chinese.get_feature_names_out()

# Convert TF-IDF matrix to a DataFrame
tfidf_df_chinese = pd.DataFrame(tfidf_matrix_chinese.toarray(), columns=feature_names_chinese)

print("TF-IDF matrix for Chinese documents created. Displaying top terms:")

# Display the top 10 most frequent (highest mean TF-IDF) terms across all Chinese documents
overall_tfidf_scores_chinese = tfidf_df_chinese.mean().sort_values(ascending=False)
print("\nTop 10 overall TF-IDF terms in Chinese documents:")
display(overall_tfidf_scores_chinese.head(10))

TF-IDF matrix for Chinese documents created. Displaying top terms:

Top 10 overall TF-IDF terms in Chinese documents:


,0
矿产,0.112245
中国,0.111324
铬矿,0.108256
资源,0.105955
关键,0.097581
中亚,0.088960
哈萨克斯坦,0.081885
合作,0.074260
我国,0.074179
国家,0.070611


### Collocation Analysis for English Documents

Collocation analysis helps identify words that frequently co-occur. For English documents, we'll use NLTK's `BigramAssocMeasures` and `BigramCollocationFinder` to find significant bigrams (two-word phrases).

### Re-running Collocation Analysis for English Documents (with Aggressive Cleaning)

Now, let's re-run the collocation analysis for English documents using the `text_cleaned` column, which has undergone more aggressive filtering.

In [ ]:
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder

# Prepare English text for collocation analysis using the aggressively cleaned text
english_words_cleaned = []
for text in english_papers_df['text_cleaned'].tolist():
    english_words_cleaned.extend(text.split())

# Initialize BigramCollocationFinder
bigram_measures = BigramAssocMeasures()
finder_english_cleaned = BigramCollocationFinder.from_words(english_words_cleaned)

# Apply a frequency filter to consider only bigrams that appear at least 10 times
finder_english_cleaned.apply_freq_filter(10)

print("Top 10 English collocations (by PMI) with aggressive cleaning:")
# Display top 10 collocations using Pointwise Mutual Information (PMI)
display(pd.Series(finder_english_cleaned.nbest(bigram_measures.pmi, 10)))

print("\nTop 10 English collocations (by likelihood ratio) with aggressive cleaning:")
# Display top 10 collocations using Likelihood Ratio
display(pd.Series(finder_english_cleaned.nbest(bigram_measures.likelihood_ratio, 10)))


Top 10 English collocations (by PMI) with aggressive cleaning:


,0
0,"(lining, wishful)"
1,"(sawingothw, blocksslabs)"
2,"(surowcw, mineralnych)"
3,"(devel, opment)"
4,"(abu, dhabi)"
5,"(eco, nomic)"
6,"(planed, sandedendjointed)"
7,"(cen, tral)"
8,"(gospodarka, surowcami)"
9,"(surowcami, mineralnymi)"



Top 10 English collocations (by likelihood ratio) with aggressive cleaning:


,0
0,"(central, asia)"
1,"(critical, minerals)"
2,"(central, asian)"
3,"(primaryprocessedsecondary, abioticnfne)"
4,"(united, states)"
5,"(raw, materials)"
6,"(rare, earth)"
7,"(supply, chains)"
8,"(semifabricated, abioticnfne)"
9,"(supply, chain)"


In [ ]:
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder

# Prepare English text for collocation analysis
# We'll re-tokenize the text after stopword removal to get individual words
english_words = []
for text in english_papers_df['text_no_stopwords'].tolist():
    english_words.extend(text.split())

# Initialize BigramCollocationFinder
bigram_measures = BigramAssocMeasures()
finder_english = BigramCollocationFinder.from_words(english_words)

# Apply a frequency filter to consider only bigrams that appear at least 10 times
finder_english.apply_freq_filter(10)

print("Top 10 English collocations (by PMI):")
# Display top 10 collocations using Pointwise Mutual Information (PMI)
display(pd.Series(finder_english.nbest(bigram_measures.pmi, 10)))

print("\nTop 10 English collocations (by likelihood ratio):")
# Display top 10 collocations using Likelihood Ratio
display(pd.Series(finder_english.nbest(bigram_measures.likelihood_ratio, 10)))

Top 10 English collocations (by PMI):


,0
0,"(wishful, thinking?)"
1,"(cut,, sawing/othw.,)"
2,"(lining, wishful)"
3,"(sawing/othw.,, blocks/slabs)"
4,"(clean., prod.)"
5,"(排, 名)"
6,"(planed,, sanded/end-jointed,)"
7,"(36(2),, 5–20)"
8,"(kuang-ho, yeh)"
9,"(cen-, tral)"



Top 10 English collocations (by likelihood ratio):


,0
0,"(central, asia)"
1,"(central, asian)"
2,"(primary/processed/secondary, abiotic-nfne)"
3,"(critical, minerals)"
4,"(et, al.)"
5,"(rare, earth)"
6,"(semi-fabricated, abiotic-nfne)"
7,"(united, states)"
8,"(raw, materials)"
9,"(supply, chain)"


### Collocation Analysis for Chinese Documents

For Chinese documents, we'll use a similar approach. Since we already have tokenized text, we can directly use it for collocation analysis.

### Re-running Collocation Analysis for Chinese Documents (with Aggressive Cleaning)

Similarly, we'll re-run the collocation analysis for Chinese documents using the `text_cleaned` column after aggressive filtering.

In [ ]:
# Prepare Chinese text for collocation analysis using the aggressively cleaned text
chinese_words_cleaned = []
for text in chinese_papers_df['text_cleaned'].tolist():
    chinese_words_cleaned.extend(text.split())

# Initialize BigramCollocationFinder for Chinese
finder_chinese_cleaned = BigramCollocationFinder.from_words(chinese_words_cleaned)

# Apply a frequency filter (e.g., bigrams appearing at least 10 times)
finder_chinese_cleaned.apply_freq_filter(10)

print("Top 10 Chinese collocations (by PMI) with aggressive cleaning:")
# Display top 10 collocations using Pointwise Mutual Information (PMI)
display(pd.Series(finder_chinese_cleaned.nbest(bigram_measures.pmi, 10)))

print("\nTop 10 Chinese collocations (by likelihood ratio) with aggressive cleaning:")
# Display top 10 collocations using Likelihood Ratio
display(pd.Series(finder_chinese_cleaned.nbest(bigram_measures.likelihood_ratio, 10)))


Top 10 Chinese collocations (by PMI) with aggressive cleaning:


,0
0,"(Commodity, Summaries)"
1,"(民心, 相通)"
2,"(EB, OL)"
3,"(视阈, 下美欧)"
4,"(超镁, 铁质)"
5,"(百年, 未有)"
6,"(White, House)"
7,"(未有, 大变局)"
8,"(中欧, 班列)"
9,"(地形, 地貌)"



Top 10 Chinese collocations (by likelihood ratio) with aggressive cleaning:


,0
0,"(关键, 矿产)"
1,"(一带, 一路)"
2,"(新能源, 汽车)"
3,"(中亚, 五国)"
4,"(一路, 倡议)"
5,"(供应, 风险)"
6,"(et, al)"
7,"(地缘, 政治)"
8,"(中亚, 各国)"
9,"(关键, 金属)"


In [ ]:
# Prepare Chinese text for collocation analysis
# The 'tokenized_text' column already contains space-separated words
chinese_words = []
for text in chinese_papers_df['tokenized_text'].tolist():
    chinese_words.extend(text.split())

# Initialize BigramCollocationFinder for Chinese
finder_chinese = BigramCollocationFinder.from_words(chinese_words)

# Apply a frequency filter (e.g., bigrams appearing at least 10 times)
finder_chinese.apply_freq_filter(10)

print("Top 10 Chinese collocations (by PMI):")
# Display top 10 collocations using Pointwise Mutual Information (PMI)
display(pd.Series(finder_chinese.nbest(bigram_measures.pmi, 10)))

print("\nTop 10 Chinese collocations (by likelihood ratio):")
# Display top 10 collocations using Likelihood Ratio
display(pd.Series(finder_chinese.nbest(bigram_measures.likelihood_ratio, 10)))

Top 10 Chinese collocations (by PMI):


,0
0,"(孙, 壮志)"
1,"(纠, 宾州)"
2,"(穆, 龙套)"
3,"(Commodity, Summaries)"
4,"(民心, 相通)"
5,"(视阈, 下美欧)"
6,"(超镁, 铁质)"
7,"(百年, 未有)"
8,"(White, House)"
9,"(中欧, 班列)"



Top 10 Chinese collocations (by likelihood ratio):


,0
0,"(２, ０)"
1,"(关键, 矿产)"
2,"(一带, 一路)"
3,"([, J)"
4,"(J, ])"
5,"(ｉ, ｎ)"
6,"(ａ, ｌ)"
7,"(*, *)"
8,"(新能源, 汽车)"
9,"(“, 一带)"


### Latent Dirichlet Allocation (LDA) for English Documents

Now, let's apply Latent Dirichlet Allocation (LDA) to the English documents. LDA is a probabilistic model that uncovers latent topics in a collection of documents. We will use `CountVectorizer` to convert the text into a bag-of-words representation, which is suitable for LDA.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

# Prepare English text for LDA (using stopword-removed text)
english_texts_lda = english_papers_df['text_no_stopwords'].tolist()

# Create a CountVectorizer for English
count_vectorizer_english = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
doc_term_matrix_english = count_vectorizer_english.fit_transform(english_texts_lda)

# Initialize and fit LDA model for English
# Let's aim for 3 topics, consistent with our BERTopic findings for English
lda_english = LatentDirichletAllocation(n_components=3, random_state=42, learning_method='online')
lda_english.fit(doc_term_matrix_english)

# Function to display top words for each topic
def display_topics(model, feature_names, num_top_words):
    for topic_idx, topic in enumerate(model.components_): # Corrected from components__ to components_
        print(f"Topic {topic_idx + 1}:")
        print(" ".join([feature_names[i] for i in topic.argsort()[:-num_top_words - 1:-1]]))

print("\nTop words for each English LDA Topic:")
display_topics(lda_english, count_vectorizer_english.get_feature_names_out(), 10)


Top words for each English LDA Topic:
Topic 1:
central kazakhstan asia critical https 2025 global eu 2024 foreign
Topic 2:
mining secondary kazakhstan primary processed steel iron semi non materials
Topic 3:
critical resource global https mining materials lithium al rare risk


### Latent Dirichlet Allocation (LDA) for Chinese Documents

Similarly, we'll apply LDA to the Chinese documents using their tokenized text. We'll use a `CountVectorizer` with a custom Chinese stopword list (or rely on `jieba` tokenization if a formal stopword list is not available) and then fit the LDA model.

In [ ]:
# Prepare Chinese text for LDA (using tokenized text)
chinese_texts_lda = chinese_papers_df['tokenized_text'].tolist()

# Create a CountVectorizer for Chinese (no default stopwords, as we already tokenized with jieba)
# You might want to provide a custom Chinese stopword list here if available
count_vectorizer_chinese = CountVectorizer(max_df=0.95, min_df=2)
doc_term_matrix_chinese = count_vectorizer_chinese.fit_transform(chinese_texts_lda)

# Initialize and fit LDA model for Chinese
# Let's aim for 3 topics, consistent with our BERTopic findings for Chinese
lda_chinese = LatentDirichletAllocation(n_components=3, random_state=42, learning_method='online')
lda_chinese.fit(doc_term_matrix_chinese)

print("\nTop words for each Chinese LDA Topic:")
display_topics(lda_chinese, count_vectorizer_chinese.get_feature_names_out(), 10)


Top words for each Chinese LDA Topic:
Topic 1:
供应商 采购 风险 供应 新能源 研究 资源 关键 管理 铬矿
Topic 2:
经济 矿产 贸易 中亚 合作 中亚国家 关键 风险 全球 影响
Topic 3:
资源 主要 经济 需求 我国 世界 中亚 战略 储量 国际
